In [1]:
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [2]:
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model = "qwen/qwen3-32b")

In [6]:
print(llm.invoke("What is the capital of France?"))

content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember from school that France's capital is Paris. I should confirm that. Maybe they're testing me or want to make sure. Let me double-check. Yeah, Paris is definitely the capital. It's a well-known fact. I don't think there's any other city that's commonly mistaken for the capital of France. Maybe they're a student studying geography or someone planning a trip. Either way, the answer is straightforward. I'll just state that Paris is the capital of France to keep it simple and accurate.\n</think>\n\nThe capital of France is Paris." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 15, 'total_tokens': 148, 'completion_time': 0.280611016, 'completion_tokens_details': None, 'prompt_time': 0.000439822, 'prompt_tokens_details': None, 'queue_time': 0.00628871, 'total_time': 0.281050838}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2',

## Data injestion

In [7]:
embedding_model = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")

In [8]:
#embedding_model.embed_query("What is the capital of France?")

In [9]:
##  data ingestion : 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\omarm\AppData\Local\Temp\ipykernel_24384\686423757.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
import os

In [11]:
os.getcwd() # get the path of the current working directory

'c:\\Users\\omarm\\document_portal\\notebook'

In [12]:
file_path = os.path.join(os.getcwd(), "data", "sample.pdf") # this is the path to the sample.pdf file in the data folder

In [ ]:
loader = PyPDFLoader(file_path) # this will load the pdf file and return a list of documents, each document will have the text of one page of the pdf file

In [ ]:
documents = loader.load() 

In [ ]:
len(documents)

27

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150 ,
    length_function = len 
) # this will split the text of each document into chunks of 500 characters with an overlap of 150 characters between chunks

In [ ]:
docs = text_splitter.split_documents(documents)

In [ ]:
len(documents) # i decided to use documents instead of docs to economise token usage

27

In [ ]:
len(docs)

258

In [ ]:
docs[0].metadata # show the info of the first document

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-02-28T01:57:46+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-02-28T01:57:46+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'c:\\Users\\omarm\\document_portal\\notebook\\data\\sample.pdf',
 'total_pages': 27,
 'page': 0,
 'page_label': '1'}

In [ ]:
docs[0].page_content # show the text of the first document

'LLaMA: Open and Efﬁcient Foundation Language Models\nHugo Touvron∗, Thibaut Lavril∗, Gautier Izacard∗, Xavier Martinet\nMarie-Anne Lachaux, Timothee Lacroix, Baptiste Rozière, Naman Goyal\nEric Hambro, Faisal Azhar, Aurelien Rodriguez, Armand Joulin\nEdouard Grave∗, Guillaume Lample∗\nMeta AI\nAbstract\nWe introduce LLaMA, a collection of founda-\ntion language models ranging from 7B to 65B\nparameters. We train our models on trillions\nof tokens, and show that it is possible to train'

In [17]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store = FAISS.from_documents(documents, embedding_model) # this will create a vector store from the documents using the embedding model

In [ ]:
vector_store

In [ ]:
len(embedding_model.embed_documents([documents[0].page_content])[0])


3072

In [ ]:
#embedding_model.embed_documents([documents[0].page_content])[0]

## Retrieval process

In [ ]:
relevant_doc = vector_store.similarity_search("llama2 finetuning benchmark experiments")

In [ ]:
relevant_doc


[Document(id='f3a407c4-6f99-4973-a096-6616b1e04c46', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-02-28T01:57:46+00:00', 'author': '', 'keywords': '', 'moddate': '2023-02-28T01:57:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Users\\omarm\\document_portal\\notebook\\data\\sample.pdf', 'total_pages': 27, 'page': 6, 'page_label': '7'}, page_content='Humanities STEM Social Sciences Other Average\nGPT-NeoX 20B 29.8 34.9 33.7 37.7 33.6\nGPT-3 175B 40.8 36.7 50.4 48.8 43.9\nGopher 280B 56.2 47.4 71.9 66.1 60.0\nChinchilla 70B 63.6 54.9 79.3 73.9 67.5\nPaLM\n8B 25.6 23.8 24.1 27.8 25.4\n62B 59.5 41.9 62.7 55.8 53.7\n540B 77.0 55.6 81.0 69.6 69.3\nLLaMA\n7B 34.0 30.5 38.3 38.1 35.1\n13B 45.0 35.8 53.8 53.3 46.9\n33B 55.8 46.0 66.7 63.4 57.8\n65B 61.8 51.7 72.9 67.4 63.4\nTable 9: Massive Multitask Language Un

In [ ]:
relevant_doc[1].metadata

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-02-28T01:57:46+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-02-28T01:57:46+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'c:\\Users\\omarm\\document_portal\\notebook\\data\\sample.pdf',
 'total_pages': 27,
 'page': 4,
 'page_label': '5'}

In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
retriever.invoke("What are the llama2 finetuning benchmark experiments?")

[Document(id='2b982e0b-d49c-47b5-bd69-f878d07cf74b', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-02-28T01:57:46+00:00', 'author': '', 'keywords': '', 'moddate': '2023-02-28T01:57:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Users\\omarm\\document_portal\\notebook\\data\\sample.pdf', 'total_pages': 27, 'page': 4, 'page_label': '5'}, page_content='HellaSwag (Zellers et al., 2019), WinoGrande (Sak-\naguchi et al., 2021), ARC easy and challenge (Clark\net al., 2018) and OpenBookQA (Mihaylov et al.,\n2018). These datasets include Cloze and Winograd\nstyle tasks, as well as multiple choice question an-\nswering. We evaluate in the zero-shot setting as\ndone in the language modeling community.\nIn Table 3, we compare with existing models\nof various sizes and report numbers from the cor-\nresponding papers

In [23]:
prompt_template = """
    Answer the question based on the following context:
    If the contest does not contain sufficient information
    say 'I do not have enough information about this '

    context: {context}
    question: {question} 
    Answer: 
"""

In [20]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser 

In [ ]:
prompt = PromptTemplate(
    template = prompt_template,
    input_variables = ["context", "question"]
)

In [ ]:
parser = StrOutputParser() # this will parse the output of the llm and return a string

In [ ]:
def format_docs(documents): 
    return "\n\n".join([doc.page_content for doc in documents]) # this will format the documents into a string with two new lines between each document

In [26]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
rag_chain = (
    {"context": retriever | format_docs , "question" : RunnablePassthrough() }
    | prompt
    | llm
    | StrOutputParser()       
             )

In [ ]:
rag_chain.invoke("tell me about the llama2 finetuning benchmark experiments?")

'<think>\nOkay, let\'s see. The user is asking about the LLaMA2 finetuning benchmark experiments. I need to check the provided context to find relevant information.\n\nLooking through the context, there\'s a section titled "4 Instruction Finetuning" which discusses finetuning models on instruction data. Specifically, they mention creating an instruct model called LLaMA-I by briefly finetuning LLaMA-65B. The results are compared in Table 10, showing improvements on MMLU. The numbers indicate that LLaMA-I 65B achieved 68.9% on MMLU, outperforming other models like OPT-IML and Flan-PaLM but not reaching the state-of-the-art of 77.4% from GPT code-davinci-002.\n\nAdditionally, in other sections like "3.2 Closed-book Question Answering" and "3.5 Code generation," there are mentions of LLaMA models\' performance with and without finetuning, but those might not directly relate to instruction finetuning benchmarks. The user\'s question specifically asks about finetuning benchmark experiments, 

## i ll build a rag that can read multiple documents : 

In [ ]:
# get the path of the pdfs , and put them in a variable called pdf_files
file_path_multiple = os.path.join(os.getcwd(), "data")
pdf_files = []
for file in os.listdir("data") : 
    if file.endswith(".pdf"):
        pdf_files.append(os.path.join(file_path_multiple, file))


In [15]:
documents = []
for pdf_file in pdf_files: # loop all the pdf files one by one
    loader = PyPDFLoader(pdf_file) # this will load the pdf file and return a list of documents, each document will have the text of one page of the pdf file
    documents.extend(loader.load())  # add the documents to the list of documents

In [16]:
len(documents)

27

In [18]:
vector_store_multiple = FAISS.from_documents(documents, embedding_model) # this will create a vector store from the documents using the embedding model

In [19]:
relevant_doc = vector_store_multiple.similarity_search("llama2 finetuning benchmark experiments")

In [21]:
retriever = vector_store_multiple.as_retriever()

In [24]:
prompt = PromptTemplate(
    template = prompt_template,
    input_variables = ["context", "question"]
)

In [25]:
parser = StrOutputParser() # this will parse the output of the llm and return a string

In [27]:
def format_docs(documents): 
    return "\n\n".join([doc.page_content for doc in documents]) # this will format the documents into a string with two new lines between each document

In [28]:
rag_chain_multiple = (
    {"context": retriever | format_docs , "question" : RunnablePassthrough() }
    | prompt
    | llm
    | StrOutputParser()       
             )

In [29]:
rag_chain_multiple.invoke("tell me about the llama2 finetuning benchmark experiments?")

'<think>\nOkay, let\'s see. The user is asking about the LLaMA2 finetuning benchmark experiments. First, I need to check the provided context for any mention of LLaMA2 or related experiments.\n\nLooking through the context, there\'s a section titled "4 Instruction Finetuning" which discusses finetuning models on instruction data. Specifically, it mentions LLaMA-I, which is an instruct model derived from LLaMA-65B. The results of this finetuning are presented in Table 10, showing that LLaMA-I 65B achieves 68.9% on MMLU, outperforming other models of similar size. \n\nThe context also compares these results with other models like OPT-IML, Flan-PaLM, and others. However, there\'s no explicit mention of "LLaMA2" in the provided text. The models referenced are LLaMA with different parameter sizes, such as LLaMA-65B. The user might be conflating LLaMA with LLaMA2, which isn\'t discussed here. \n\nSince the context doesn\'t provide information on LLaMA2 specifically, but does talk about LLaMA